In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
"""
=========================================================
ROGII Geosteering Prediction: FINAL SUBMISSION PIPELINE
=========================================================
1. 爆速実験で勝ち残った「黄金の6特徴量（Golden Set）」のみを使用
2. 多重共線性を排除し、AIの迷いをゼロに
3. LightGBM と CatBoost による 5-Fold アンサンブル学習
4. Savitzky-Golay フィルタによる地層の物理的連続性の復元
=========================================================
"""

import os
import glob
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor
from scipy.signal import savgol_filter
from sklearn.model_selection import GroupKFold
from sklearn.metrics import root_mean_squared_error
import warnings

warnings.filterwarnings('ignore')

print("🚀 最終本番パイプライン（Golden Set + Ensemble + SG Filter）を開始します...\n")

DATA_DIR = '/kaggle/input/competitions/rogii-wellbore-geology-prediction'

# ==========================================
# 1. データ読み込みと「黄金の6特徴量」の作成
# ==========================================
def process_golden_data(directory, is_test=False):
    files = glob.glob(os.path.join(directory, '*__horizontal_well.csv'))
    all_data = []
    
    for file_path in files:
        w_id = os.path.basename(file_path).split('__')[0]
        df = pd.read_csv(file_path)
        df['well_id'] = w_id
        df['row_order'] = np.arange(len(df))
        
        if df['TVT_input'].isna().all():
            continue
            
        last_valid_idx = df['TVT_input'].last_valid_index()
        last_tvt = float(df.loc[last_valid_idx, 'TVT_input'])
        last_z = float(df.loc[last_valid_idx, 'Z'])
        last_md = float(df.loc[last_valid_idx, 'MD'])
        
        # 基準点との差分（計算用）
        df['Z_since'] = df['Z'].values - last_z
        md_since = df['MD'].values - last_md
        
        # --- 👑 黄金の6特徴量（The Golden Set） ---
        
        # 1. 空間の歪み（新王者）
        df['Z_MD_Interaction'] = df['Z_since'].values * md_since
        
        # 2. 曲がり具合（総カーブ量）
        df['Deviation_absolute'] = df['MD'].values - df['Z'].values
        
        # 3. 井戸内の深さ進行度（Local Scaling）
        df['Z_local_norm'] = (df['Z'].values - df['Z'].min()) / (df['Z'].max() - df['Z'].min() + 1e-8)
        
        # 4. 蛇行率（無駄走り）
        df['Tortuosity'] = np.where(df['Z_since'].values != 0, md_since / df['Z_since'].values, 0)
        
        # 5. 累積ガンマ線（唯一生き残った地質センサー）
        df['GR_cumsum'] = df['GR'].cumsum()
        last_gr_cumsum = float(df.loc[last_valid_idx, 'GR_cumsum'])
        df['GR_cumsum_since'] = df['GR_cumsum'].values - last_gr_cumsum
        
        # (Z_since はそのまま6つ目として使用)

        if not is_test:
            # 予測ターゲット（基準点からのTVTの変化量）
            df['target_diff'] = df['TVT'].values - last_tvt
            
        if is_test:
            df['id'] = w_id + '_' + df.index.astype(str)
            df['is_unknown'] = df.index > last_valid_idx
            df['last_known_tvt'] = last_tvt
            
        all_data.append(df)
    return pd.concat(all_data, ignore_index=True)

print("⏳ 1/4: データを読み込み、Golden Set特徴量を作成中...")
train_df = process_golden_data(f'{DATA_DIR}/train', is_test=False)
test_df = process_golden_data(f'{DATA_DIR}/test', is_test=True)


# ==========================================
# 2. 本番用アンサンブル学習 (LightGBM + CatBoost)
# ==========================================
print("\n🧠 2/4: 本番用アンサンブル学習 (5-Fold CV) を開始します...")
print("※数分〜十数分かかります。AIの予測をブレンド中です！\n")

FEATURES = [
    'Z_MD_Interaction', 
    'Z_since', 
    'Deviation_absolute', 
    'Z_local_norm', 
    'Tortuosity', 
    'GR_cumsum_since'
]
TARGET = 'target_diff'
GROUP = 'well_id'

X = train_df[FEATURES]
y = train_df[TARGET]
groups = train_df[GROUP]

gkf = GroupKFold(n_splits=5)
oof_preds_lgb = np.zeros(len(train_df))
oof_preds_cat = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))

lgb_params = {
    'n_estimators': 500,
    'learning_rate': 0.05,
    'max_depth': 6,
    'random_state': 42,
    'n_jobs': -1
}

cat_params = {
    'iterations': 500,
    'learning_rate': 0.05,
    'depth': 6,
    'random_state': 42,
    'verbose': False
}

for fold, (train_idx, valid_idx) in enumerate(gkf.split(X, y, groups)):
    X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
    X_valid_fold, y_valid_fold = X.iloc[valid_idx], y.iloc[valid_idx]
    
    # --- LightGBM ---
    model_lgb = lgb.LGBMRegressor(**lgb_params)
    model_lgb.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_valid_fold, y_valid_fold)],
        callbacks=[lgb.early_stopping(stopping_rounds=40, verbose=False)]
    )
    valid_pred_lgb = model_lgb.predict(X_valid_fold)
    oof_preds_lgb[valid_idx] = valid_pred_lgb
    
    # --- CatBoost ---
    model_cat = CatBoostRegressor(**cat_params)
    model_cat.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_valid_fold, y_valid_fold)],
        early_stopping_rounds=40
    )
    valid_pred_cat = model_cat.predict(X_valid_fold)
    oof_preds_cat[valid_idx] = valid_pred_cat
    
    # --- アンサンブル (平均) ---
    pred_lgb_test = model_lgb.predict(test_df[FEATURES])
    pred_cat_test = model_cat.predict(test_df[FEATURES])
    test_preds += ((pred_lgb_test + pred_cat_test) / 2) / gkf.n_splits
    
    fold_rmse = root_mean_squared_error(y_valid_fold, (valid_pred_lgb + valid_pred_cat) / 2)
    print(f"--- Fold {fold + 1} Ensemble RMSE: {fold_rmse:.4f}")

oof_preds_ensemble = (oof_preds_lgb + oof_preds_cat) / 2
overall_rmse = root_mean_squared_error(y, oof_preds_ensemble)
print("========================================")
print(f"🏆 Overall Ensemble OOF RMSE: {overall_rmse:.4f}")
print("========================================")


# ==========================================
# 3. 予測の復元と平滑化（Savitzky-Golay）
# ==========================================
print("\n📝 3/4: TVTの復元とSavitzky-Golayフィルタによる平滑化を実行中...")

test_df['pred_diff'] = test_preds
# 最後の既知のTVTに、予測した変化量を足して復元
test_df['pred_tvt'] = test_df['last_known_tvt'] + test_df['pred_diff']

def apply_smoothing(df):
    smoothed_preds = []
    for w_id, g in df.groupby('well_id', sort=False):
        v = g['pred_tvt'].values
        n = len(v)
        wl = min(17, n)
        if wl % 2 == 0: wl -= 1
        if wl >= 5: 
            v = savgol_filter(v, wl, 3)
        g['pred_tvt_smooth'] = v
        smoothed_preds.append(g)
    return pd.concat(smoothed_preds)

test_df = apply_smoothing(test_df)


# ==========================================
# 4. 提出用ファイルの作成
# ==========================================
print("📝 4/4: submission.csv を作成しています...")

sample_sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
sample_sub['well_id'] = sample_sub['id'].apply(lambda x: x.split('_')[0])
sample_sub['row_order'] = sample_sub.groupby('well_id').cumcount()

unknown_test = test_df[test_df['is_unknown'] == True].copy()
unknown_test['row_order'] = unknown_test.groupby('well_id').cumcount()

sub_fixed = sample_sub.merge(
    unknown_test[['well_id', 'row_order', 'pred_tvt_smooth']], 
    on=['well_id', 'row_order'], 
    how='left'
)

# 欠損値のフォールバック処理（直前の値で埋める、無理なら11500）
sub_fixed['pred_tvt_smooth'] = sub_fixed.groupby('well_id')['pred_tvt_smooth'].fillna(method='ffill')
sub_fixed['pred_tvt_smooth'] = sub_fixed['pred_tvt_smooth'].fillna(11500) 

sub_final = pd.DataFrame({'id': sub_fixed['id'], 'tvt': sub_fixed['pred_tvt_smooth']})
sub_final.to_csv('submission.csv', index=False)

print("========================================")
print("🎉 'submission.csv' が完成しました！そのままSubmitへ進んでください！")
print("========================================")